# From Voice to Text and Back
## ASR, CTC, WER, and TTS Project

**A single-notebook version of the full runnable project.** This notebook includes setup, explanations, audio loading, waveform and log-Mel analysis, Wav2Vec2/CTC, Whisper ASR, a CTC decoding example, WER comparison, TTS, output saving, interpretation, references, and a timed presentation script.

> **Educational scope:** This demonstrates speech-processing techniques on one sample. It is not a clinical assessment, a stutter detector, or evidence about any speaker. Do not redistribute voice recordings or derived transcripts without the speaker's informed permission and any necessary rights.

### Contents
1. [Setup and configuration](#setup-and-configuration)
2. [Audio and acoustic features](#audio-and-acoustic-features)
3. [ASR models](#automatic-speech-recognition-asr)
4. [CTC decoding](#ctc-connectionist-temporal-classification)
5. [WER evaluation](#word-error-rate-wer)
6. [Text-to-speech and outputs](#text-to-speech-tts-and-saved-outputs)
7. [Interpretation, limitations, and demo](#interpretation-limitations-and-demonstration)

**Run from top to bottom.** In VS Code/Jupyter, open this file at the repository root and use **Run All**. In Google Colab, upload this notebook and run the setup cell; the audio-loading cell prompts you to upload the recording if the repository audio path is unavailable. Model weights are downloaded on first use.

**Pipeline:** Speech recording → waveform/features → log-Mel visualization → Wav2Vec2 + CTC / Whisper → transcripts → WER against a checked reference → TTS audio.

## Current run snapshot

- Input: `audio/stutter_current.mp3` (140.35 seconds, 48 kHz source; mono analysis).
- ASR outputs: refreshed Wav2Vec2/CTC and Whisper transcripts in `outputs/`.
- TTS output: `outputs/tts_output.wav` (mono PCM, 22,050 Hz, about 37.65 seconds).
- Current WER CSV: 115-word provisional reference copied from Whisper. Wav2Vec2 differs by 39.13%; Whisper is 0% because it is the reference. These are agreement scores, not verified accuracy scores.

The reference must be independently checked against the recording before presenting WER as model accuracy.


## Setup and configuration

### Run locally (PowerShell)

From the project root, create/activate the existing virtual environment and install dependencies:

```powershell
py -m venv .venv
.\.venv\Scripts\Activate.ps1
python -m pip install --upgrade pip
python -m pip install -r requirements.txt
```

Then open `stutter_speech_pipeline.ipynb` and run all cells. To run this notebook by itself without the repository's requirements file, run the next cell.

### Run in Google Colab

Run the package-install cell, then run all cells. If the audio is not already present at `audio/stutter_current.mp3`, the notebook will open Colab's upload control. For a different recording, change `AUDIO_PATH` below or upload it when prompted.

### Software, compute, and privacy notes

- Python 3.10+ is recommended. PyTorch and Hugging Face libraries download sizable model files and may need several gigabytes of disk space; CPU inference works but can be slow.
- gTTS needs internet access to synthesize speech.
- The default source in this repository is `audio/stutter_current.mp3`; generated figures, transcripts, WER CSV, and TTS audio are saved under `outputs/`.
- The current reference is a provisional copy of the Whisper output. Replace it with an independently human-verified transcript before interpreting WER as accuracy.
- Whisper is an encoder-decoder model; it does **not** use CTC. CTC applies to the Wav2Vec2 path here.

In [ ]:
%pip install -q ipython ipykernel librosa matplotlib numpy soundfile torch transformers faster-whisper jiwer pandas gTTS

In [ ]:
from pathlib import Path
import sys

import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np

# Run locally from the repository root; edit this path for a different recording.
AUDIO_PATH = Path("audio/stutter_current.mp3")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not AUDIO_PATH.is_file():
    if "google.colab" in sys.modules:
        from google.colab import files
        uploaded = files.upload()
        if not uploaded:
            raise FileNotFoundError("No audio file was uploaded.")
        AUDIO_PATH = Path(next(iter(uploaded)))
    else:
        raise FileNotFoundError(
            f"Audio not found: {AUDIO_PATH.resolve()}. Set AUDIO_PATH to your recording. "
            "If using a notebook outside the project root, set the full audio path."
        )

print(f"Audio input: {AUDIO_PATH.resolve()}")
print(f"Outputs:    {OUTPUT_DIR.resolve()}")

## Audio and acoustic features

The waveform shows signal amplitude over time. A Mel spectrogram summarizes short-time frequency energy on a perceptually spaced frequency scale; converting its power values to decibels gives the log-Mel visualization. These plots describe the signal—they do not diagnose stuttering.

In [ ]:
from IPython.display import Audio, display

audio, source_sr = librosa.load(AUDIO_PATH, sr=None, mono=True)
if audio.size == 0:
    raise ValueError(f"The audio file contains no samples: {AUDIO_PATH}")
duration_seconds = len(audio) / source_sr
audio_16k = librosa.resample(audio, orig_sr=source_sr, target_sr=16000)

print(f"Sample rate: {source_sr:,} Hz")
print(f"Duration:    {duration_seconds:.2f} seconds")
print(f"Samples:     {len(audio):,}")
display(Audio(data=audio, rate=source_sr))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9))
librosa.display.waveshow(audio, sr=source_sr, ax=axes[0])
axes[0].set(title="Speech waveform", xlabel="Time (seconds)", ylabel="Amplitude")

n_fft = 2048
hop_length = 512
n_mels = 128
mel_power = librosa.feature.melspectrogram(
    y=audio, sr=source_sr, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels
)
log_mel_db = librosa.power_to_db(mel_power, ref=np.max)
image = librosa.display.specshow(
    log_mel_db, sr=source_sr, hop_length=hop_length,
    x_axis="time", y_axis="mel", ax=axes[1]
)
axes[1].set(title="Log-Mel spectrogram", xlabel="Time (seconds)", ylabel="Mel frequency")
fig.colorbar(image, ax=axes[1], format="%+2.0f dB")
fig.tight_layout()
waveform_spectrogram_path = OUTPUT_DIR / "notebook_waveform_spectrogram.png"
fig.savefig(waveform_spectrogram_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Log-Mel feature matrix: {log_mel_db.shape} (Mel bands × frames)")
print(f"Combined plot saved: {waveform_spectrogram_path}")

### Feature extraction: what the settings mean

- **Mono audio:** combines the channels for this single-speaker demonstration.
- **16 kHz resampling:** used as the Wav2Vec2 model input rate.
- **FFT window 2048, hop 512, 128 Mel bands:** parameters for the visualization, not a learned diagnosis.
- **Log power:** expresses relative spectral energy in decibels so quiet and strong regions are easier to inspect.

## Automatic speech recognition (ASR)

### Wav2Vec2 with CTC

The pretrained `facebook/wav2vec2-base-960h` model takes normalized waveform samples at 16 kHz. Its CTC head scores vocabulary tokens across audio frames. Greedy decoding selects the most likely token at each frame; the processor merges repeated labels and removes the CTC blank to obtain text.

In [ ]:
import torch
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

WAV2VEC_MODEL_NAME = "facebook/wav2vec2-base-960h"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Loading {WAV2VEC_MODEL_NAME} on {device}...")
processor = Wav2Vec2Processor.from_pretrained(WAV2VEC_MODEL_NAME)
wav2vec_model = Wav2Vec2ForCTC.from_pretrained(WAV2VEC_MODEL_NAME).to(device)
wav2vec_model.eval()
model_inputs = processor(
    audio_16k, sampling_rate=16000, return_tensors="pt", padding=True
).input_values.to(device)
with torch.inference_mode():
    logits = wav2vec_model(model_inputs).logits
predicted_ids = logits.argmax(dim=-1)
wav2vec_text = processor.batch_decode(predicted_ids)[0].strip()
if not wav2vec_text:
    raise RuntimeError("Wav2Vec2 returned an empty transcript; inspect the audio and model output.")
print("Wav2Vec2 / CTC transcript:")
print(wav2vec_text)
wav2vec_transcript_path = OUTPUT_DIR / "asr_transcription.txt"
wav2vec_transcript_path.write_text(wav2vec_text + "\n", encoding="utf-8")
print(f"Saved: {wav2vec_transcript_path}")

### Whisper

Whisper-base is a separate encoder-decoder Transformer: it encodes log-Mel audio features and generates text tokens autoregressively. This implementation uses faster-whisper with CPU/int8 for broad compatibility (change settings only if you understand the available hardware). It is not a CTC model.

Some PyAV versions removed the optional `metadata_errors` keyword expected by certain faster-whisper versions. The small compatibility adapter below strips only that obsolete keyword.

In [ ]:
import av

# Compatibility with PyAV versions that no longer accept metadata_errors.
_original_av_open = av.open
def _av_open_compat(*args, **kwargs):
    kwargs.pop("metadata_errors", None)
    return _original_av_open(*args, **kwargs)
av.open = _av_open_compat

from faster_whisper import WhisperModel

WHISPER_MODEL_NAME = "base"
print(f"Loading faster-whisper {WHISPER_MODEL_NAME} (CPU/int8)...")
whisper_model = WhisperModel(WHISPER_MODEL_NAME, device="cpu", compute_type="int8")
segments, whisper_info = whisper_model.transcribe(
    str(AUDIO_PATH), language="en", beam_size=5
)
segment_texts = []
for segment in segments:
    text = segment.text.strip()
    if text:
        segment_texts.append(text)
        print(f"[{segment.start:.2f}s–{segment.end:.2f}s] {text}")
whisper_text = " ".join(segment_texts).strip()
if not whisper_text:
    raise RuntimeError("Whisper returned an empty transcript; inspect the audio and model output.")
print(f"Detected language: {whisper_info.language} (probability {whisper_info.language_probability:.2f})")
print("\nWhisper transcript:")
print(whisper_text)
whisper_transcript_path = OUTPUT_DIR / "whisper_transcription.txt"
whisper_transcript_path.write_text(whisper_text + "\n", encoding="utf-8")
print(f"Saved: {whisper_transcript_path}")

### Architecture overview

Two ASR routes are compared. Waveform samples go to Wav2Vec2 and its CTC decoder; audio features go to Whisper's encoder-decoder. Both produce text for optional WER evaluation. A selected transcript can then be synthesized as speech.

In [ ]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(14, 5.5))
ax.set_xlim(0, 14)
ax.set_ylim(0, 6)
ax.axis("off")

def add_box(x, y, width, height, title, detail, color):
    box = FancyBboxPatch((x, y), width, height, boxstyle="round,pad=0.08",
                         facecolor=color, edgecolor="#334155", linewidth=1.4)
    ax.add_patch(box)
    ax.text(x + width / 2, y + height * 0.63, title, ha="center", va="center",
            fontsize=10, fontweight="bold", wrap=True)
    ax.text(x + width / 2, y + height * 0.28, detail, ha="center", va="center",
            fontsize=8, wrap=True)

def add_arrow(x1, y1, x2, y2):
    ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle="-|>",
                                 mutation_scale=14, color="#475569", linewidth=1.3))

add_box(0.2, 2.1, 1.8, 1.4, "Speech audio", "MP3 / waveform", "#e0f2fe")
add_box(2.6, 3.7, 2.4, 1.3, "Wav2Vec2", "waveform → token scores", "#ede9fe")
add_box(5.7, 3.7, 2.0, 1.3, "CTC decoder", "collapse repeats + blanks", "#f3e8ff")
add_box(2.6, 0.7, 2.4, 1.3, "Log-Mel features", "short-time spectrum", "#dcfce7")
add_box(5.7, 0.7, 2.0, 1.3, "Whisper", "encoder-decoder ASR", "#fef3c7")
add_box(8.4, 2.1, 1.8, 1.4, "Transcript", "human-verified ref for WER", "#ffedd5")
add_box(11.0, 3.7, 2.4, 1.3, "WER comparison", "hits / substitutions / deletions / insertions", "#fee2e2")
add_box(11.0, 0.7, 2.4, 1.3, "Text-to-speech", "synthesized audio", "#dbeafe")
add_arrow(2.0, 2.9, 2.55, 4.25)
add_arrow(5.0, 4.35, 5.65, 4.35)
add_arrow(7.75, 4.35, 8.35, 3.2)
add_arrow(2.0, 2.6, 2.55, 1.35)
add_arrow(5.0, 1.35, 5.65, 1.35)
add_arrow(7.75, 1.35, 8.35, 2.4)
add_arrow(10.2, 2.8, 10.95, 4.0)
add_arrow(10.2, 2.6, 10.95, 1.35)
ax.set_title("Speech → Features → ASR → Text → Evaluation / TTS", fontsize=14, pad=12)
fig.tight_layout()
architecture_path = OUTPUT_DIR / "asr_architecture_notebook.png"
fig.savefig(architecture_path, dpi=180, bbox_inches="tight")
plt.show()
print(f"Notebook architecture diagram saved: {architecture_path}")

## CTC (Connectionist Temporal Classification)

CTC trains sequence models when the precise alignment between audio frames and transcript characters is unknown. A frame-level path can contain a special blank token and repeated labels. The greedy decoder's collapse operation (B) first merges consecutive repeats, then removes blanks. The order is important: `A, A` becomes `A`, while `A, blank, A` becomes `AA`.

For a fixed frame-token path this notebook demonstrates greedy collapse; production CTC decoding can instead search multiple paths (beam search) and optionally use a language model.

In [ ]:
BLANK = 0
TOKEN_LABELS = {1: "A", 2: "B"}
frame_tokens = [BLANK, 1, 1, BLANK, 1, 2, 2, BLANK]

collapsed = []
previous = None
for token in frame_tokens:
    if token != previous:
        collapsed.append(token)
    previous = token
decoded_tokens = [token for token in collapsed if token != BLANK]
decoded_text = "".join(TOKEN_LABELS[token] for token in decoded_tokens)
print("Frame tokens (0 is blank):", frame_tokens)
print("After merging adjacent repeats:", collapsed)
print("After removing blanks:", decoded_tokens)
print("Decoded text:", decoded_text)
assert decoded_text == "AAB"
print("Check passed: A, blank, A remains AA; adjacent A, A collapses to A.")

## Word error rate (WER)

WER compares a hypothesis with a reference after consistent normalization:



[\mathrm{WER} = \frac{S + D + I}{N}\]



where (S) is substitutions, (D) deletions, (I) insertions, and (N) the reference word count. Lower values indicate closer word sequences for that reference. WER can exceed 100%. It does not measure clinical severity or whether a transcript preserved disfluencies.

**Reference protocol:** Listen to the recording and manually verify the reference, including decisions on repetitions, fillers, and punctuation. Enter the verified transcript in the next cell and set `REFERENCE_IS_VERIFIED = True`. By default, this notebook reads the repository's reference file if available but labels the score **PROVISIONAL**; that file has not been independently verified. The current reference was modeled closely on Whisper output, so comparing Whisper against it is biased.

In [ ]:
import csv
import re
from jiwer import process_words, wer

# Replace this with a human-verified transcript before treating WER as accuracy.
REFERENCE_TEXT = ""
REFERENCE_IS_VERIFIED = False

if REFERENCE_TEXT.strip():
    reference_raw = REFERENCE_TEXT.strip()
    reference_source = "notebook entry"
else:
    reference_file = OUTPUT_DIR / "reference_transcript.txt"
    if not reference_file.is_file():
        raise FileNotFoundError(f"Reference transcript not found: {reference_file}")
    reference_raw = reference_file.read_text(encoding="utf-8").strip()
    reference_source = str(reference_file)
if not reference_raw:
    raise ValueError("Reference transcript is empty.")

def normalize_transcript(text):
    text = re.sub(r"[^\w\s']", " ", text.lower())
    return " ".join(text.split())

reference = normalize_transcript(reference_raw)
whisper_normalized = normalize_transcript(whisper_text)
if REFERENCE_IS_VERIFIED:
    reference_status = "human_verified"
elif reference == whisper_normalized:
    reference_status = "provisional_whisper_copy_unverified"
else:
    reference_status = "unverified_reference"

hypotheses = {"Wav2Vec2 CTC": wav2vec_text, "Whisper": whisper_text}
wer_rows = []
for model_name, hypothesis_raw in hypotheses.items():
    hypothesis = normalize_transcript(hypothesis_raw)
    alignment = process_words(reference, hypothesis)
    wer_rows.append({
        "model": model_name,
        "reference_words": len(reference.split()),
        "hits": alignment.hits,
        "substitutions": alignment.substitutions,
        "deletions": alignment.deletions,
        "insertions": alignment.insertions,
        "wer": wer(reference, hypothesis),
        "reference_status": reference_status,
    })

wer_csv_path = OUTPUT_DIR / "wer_results.csv"
with wer_csv_path.open("w", encoding="utf-8", newline="") as result_file:
    writer = csv.DictWriter(result_file, fieldnames=wer_rows[0].keys())
    writer.writeheader()
    writer.writerows(wer_rows)

import pandas as pd
wer_table = pd.DataFrame(wer_rows)
wer_table["wer_percent"] = wer_table["wer"].map(lambda score: f"{score:.2%}")
display(wer_table[["model", "reference_words", "hits", "substitutions",
                   "deletions", "insertions", "wer_percent", "reference_status"]])
print(f"Reference source: {reference_source}")
print(f"Reference status: {reference_status}")
print(f"Saved metrics: {wer_csv_path}")
if reference_status == "provisional_whisper_copy_unverified":
    print("WARNING: Whisper is its own reference, so its 0% is by construction; scores are not verified accuracy.")
elif reference_status != "human_verified":
    print("WARNING: Reference is unverified; do not report these scores as accuracy.")


## Text-to-speech (TTS) and saved outputs

The notebook can synthesize the Whisper transcript using Google Text-to-Speech (gTTS), save an MP3, and play it. The current project WAV was generated with Windows System.Speech. Internet access is required. The synthetic voice does not reproduce the source speaker's identity, delivery, or stutter. A Windows-only System.Speech WAV version from the original project is provided as an optional documented alternative below.

In [ ]:
from gtts import gTTS
from IPython.display import Audio, display

TTS_TEXT = whisper_text
if not TTS_TEXT.strip():
    raise ValueError("TTS text is empty; run Whisper successfully or set TTS_TEXT manually.")
tts_path = OUTPUT_DIR / "notebook_tts_output.mp3"
gTTS(text=TTS_TEXT, lang="en").save(str(tts_path))
print(f"Synthesized speech saved: {tts_path}")
display(Audio(filename=str(tts_path)))

### Optional: produce a WAV with Windows System.Speech

The original project also used Windows' built-in `System.Speech` synthesizer to create `outputs/tts_output.wav`. This optional Python cell reproduces that path on Windows without requiring the separate `tts.ps1` script. It is skipped on other operating systems. This voice is synthesized and is not the source speaker.

In [ ]:
import base64
import os
import subprocess

if os.name == "nt":
    windows_tts_path = OUTPUT_DIR / "tts_output.wav"
    output_b64 = base64.b64encode(str(windows_tts_path).encode("utf-8")).decode("ascii")
    text_b64 = base64.b64encode(TTS_TEXT.encode("utf-8")).decode("ascii")
    powershell_code = (
        f"$outputPath = [Text.Encoding]::UTF8.GetString([Convert]::FromBase64String('{output_b64}')); "
        f"$speechText = [Text.Encoding]::UTF8.GetString([Convert]::FromBase64String('{text_b64}')); "
        "Add-Type -AssemblyName System.Speech; "
        "$s = New-Object System.Speech.Synthesis.SpeechSynthesizer; "
        "try { $s.SetOutputToWaveFile($outputPath); $s.Speak($speechText) } "
        "finally { $s.Dispose() }"
    )
    encoded_command = base64.b64encode(powershell_code.encode("utf-16le")).decode("ascii")
    subprocess.run(
        ["powershell", "-NoProfile", "-EncodedCommand", encoded_command],
        check=True,
    )
    print(f"Windows System.Speech audio saved: {windows_tts_path}")
else:
    print("Windows System.Speech WAV is unavailable on this OS; use the gTTS MP3 above.")

### Files generated by the notebook

| Output | Contents |
|---|---|
| `outputs/notebook_waveform_spectrogram.png` | Waveform and log-Mel plot |
| `outputs/asr_transcription.txt` | Wav2Vec2/CTC hypothesis |
| `outputs/whisper_transcription.txt` | Whisper hypothesis |
| `outputs/wer_results.csv` | WER counts, scores, and reference-status label |
| `outputs/notebook_tts_output.mp3` | gTTS audio generated from selected transcript |
| `outputs/tts_output.wav` | Optional Windows System.Speech audio |
| `outputs/asr_architecture_notebook.png` | Pipeline diagram rendered in this notebook |

Each output cell also prints its saved path. Keep the recording and derived data private unless you have permission to share them.

## Interpretation, limitations, and demonstration

### How to interpret the comparison

- ASR can normalize, omit, or alter repetitions and fillers; inspect each transcript against the audio.
- A low WER on one sample is not a general claim about the model or about people who stutter.
- If the reference is unverified or derived from one of the hypotheses, the comparison is not a fair accuracy benchmark. Report the reference source and verification status with any score.
- The log-Mel plot is a visualization, not a classifier. This project has no model trained to detect stuttering and must not be used for diagnosis.
- The two systems use different architectures and inputs. Wav2Vec2 uses CTC here; Whisper does not.

### Three-minute demonstration script

**0:00–0:25 — Objective and recording.** “This project follows one sample of speech through audio analysis, recognition, evaluation, and speech synthesis. It is an educational demonstration, not a diagnostic tool.”

**0:25–0:55 — Waveform and log-Mel.** “The waveform displays amplitude over time. The log-Mel spectrogram shows frequency energy on a perceptually spaced scale. They visualize the signal but cannot diagnose a speech disorder.”

**0:55–1:25 — ASR paths.** “Wav2Vec2 takes waveform samples and predicts token scores; CTC decodes its output. Whisper uses an encoder-decoder to generate text from audio features and does not use CTC.”

**1:25–1:55 — CTC.** “CTC allows model frames to align flexibly to text. Greedy decoding merges adjacent repeats and removes blanks. A blank between two identical letters lets both letters remain.”

**1:55–2:25 — WER.** “Word error rate is substitutions plus deletions plus insertions, divided by reference words. The current comparison uses Whisper as its own provisional reference, so its 0% is by construction; verify an independent transcript before interpreting accuracy.”

**2:25–2:50 — TTS.** “I synthesize a transcript into a new audio file. The generated voice is not the original speaker and does not preserve the original stutter.”

**2:50–3:00 — Wrap-up.** “The notebook includes the complete reproducible workflow. The key next step for a fair accuracy comparison is a carefully verified reference transcript.”

### References

- Baevski et al., [wav2vec 2.0: A Framework for Self-Supervised Learning of Speech Representations](https://arxiv.org/abs/2006.11477).
- Graves et al., [Connectionist Temporal Classification](https://www.cs.toronto.edu/~graves/icml_2006.pdf).
- Radford et al., [Robust Speech Recognition via Large-Scale Weak Supervision (Whisper)](https://arxiv.org/abs/2212.04356).
- [Hugging Face Wav2Vec2 documentation](https://huggingface.co/docs/transformers/model_doc/wav2vec2).
- [faster-whisper project](https://github.com/SYSTRAN/faster-whisper).
- [JiWER documentation](https://jitsi.github.io/jiwer/).
- [gTTS documentation](https://gtts.readthedocs.io/).

### Reproducibility checklist

1. Record which audio file and model versions were used.
2. Run all cells from top to bottom; retain the printed model/device and audio metadata.
3. Listen to both hypotheses and the audio. Correct a verbatim reference independently.
4. Set `REFERENCE_IS_VERIFIED = True` only after checking it; rerun the WER cell and retain the CSV.
5. Regenerate the TTS output if the selected transcript changes.
6. Include consent/privacy and the one-recording limitation when presenting results.